In [2]:
import torch
import os
import inspect
import time
import torch.distributed as dist
from dataclasses import dataclass
from torch.nn.parallel import DistributedDataParallel as DDP

from gpt import GPT
from DataLoader import DataLoader

# torchrun --standalone --nproc-per-node=8 file.py

os.environ["TORCHINDUCTOR_CACHE_DIR"] = ".inductor_cache"
start_time = time.time()
time_limit = 3600*4

@dataclass
class GPTConfig:
    batch_size: int = 4
    block_size: int = 256
    vocab_size: int = 50257 # tokenizer.n_vocab # 50257, 50000 merges + 256 byte + <endoftext>
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    lr: float = 3e-4

# -------------------------------------------------------------------------------

ddp = int(os.environ.get('RANK', -1)) != -1 # using ddp torch run?
if ddp:
    assert torch.cuda.is_available(), "you need CUDA for DDP"
    dist.init_process_group(backend='nccl')
    ddp_rank = int(os.environ['RANK']) # which gpu 0,1,2,..
    ddp_local_rank = int(os.environ['LOCAL_RANK']) # multinode
    ddp_world_size = int(os.environ['WORLD_SIZE']) # total gpus
    device = f"cuda:{ddp_local_rank}"
    torch.cuda.set_device(device=device)
else:
    ddp_rank = 0
    ddp_local_rank = 0
    ddp_world_size = 1

    device = 'cpu'
    if torch.cuda.is_available():
        device = 'cuda'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = 'mps'
    print(device)

master_process = ddp_rank == 0
if master_process:
    print(ddp_world_size)


cuda
1


In [3]:
# -------------------------------------------------------------------------------
ckpt_path = "cleaned10k.pt"
def load_checkpoint(ckpt_path, model):
    if not (os.path.exists(ckpt_path)):
        return 1

    checkpoint = torch.load(ckpt_path, weights_only=True)
    model.load_state_dict(checkpoint['model'])

    return checkpoint['step']

def save_checkpoint(raw_model, step):
    checkpoint = {
        'model': raw_model.state_dict(),
        'step': step
    }
    torch.save(checkpoint, f"ddp_{step}.pt")
# -------------------------------------------------------------------------------
max_iters = 10000
save_interval = 3
tokens_per_step = 524288
grad_accum_steps = tokens_per_step // (GPTConfig().batch_size * GPTConfig().block_size * ddp_world_size)
# grad_accum_steps = 16


loader = DataLoader(B=GPTConfig.batch_size, T=GPTConfig().block_size, gpu_rank=ddp_rank, total_gpus=ddp_world_size) # GPTConfig.block_size
loader.next_batch('train')

model = GPT(GPTConfig(vocab_size=50304))
model = model.to(device=device)

if torch.cuda.is_available() and torch.cuda.get_device_capability(device)[0] >= 7:
    model = torch.compile(model)
else:
    print("Triton only supports devices of CUDA Capability >= 7.0")


tokens: 304222
1 epoch is 297 iters
Triton only supports devices of CUDA Capability >= 7.0


In [4]:
left_off_step = load_checkpoint(ckpt_path=ckpt_path, model=model)

In [5]:
fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
used_fused = fused_available and 'cuda' in device
optimizer = torch.optim.AdamW(model.parameters(), lr=model.config.lr, fused=used_fused)


In [6]:
for step in range(10):
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)
    Xb, Yb = loader.next_batch('train')
    Xb, Yb = Xb.to(device), Yb.to(device)

    with torch.autocast(device_type=device, dtype=torch.bfloat16): # slower on pascal arch
        logits, loss = model(Xb, Yb)

    loss.backward() # changed due to ddp wrap
    optimizer.step() # since all gpu's have the same gradients, we are "sync" with the weights after we update as well

    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    torch.cuda.synchronize()
    t1 = time.time()
    dt = (t1 - t0)
    tokens = (loader.B * loader.T * grad_accum_steps * ddp_world_size) / dt
    print(f"step: {step}, loss: {loss.item():.4f}, norm: {norm:.2f}, time: {1000 * dt:.2f} ms, tok/sec: {tokens:.2f}")
    t0 = time.time()

step: 0, loss: 0.0033, norm: 0.30, time: 532.45 ms, tok/sec: 984677.99
step: 1, loss: 0.0034, norm: 0.23, time: 214.03 ms, tok/sec: 2449545.64
step: 2, loss: 0.0176, norm: 1.01, time: 209.00 ms, tok/sec: 2508553.70
step: 3, loss: 0.0283, norm: 0.18, time: 210.00 ms, tok/sec: 2496563.72
step: 4, loss: 0.0044, norm: 0.41, time: 207.00 ms, tok/sec: 2532794.98
step: 5, loss: 0.0145, norm: 2.15, time: 205.48 ms, tok/sec: 2551488.42
step: 6, loss: 0.0317, norm: 0.34, time: 208.08 ms, tok/sec: 2519651.42
step: 7, loss: 0.0526, norm: 1.02, time: 206.22 ms, tok/sec: 2542343.92
step: 8, loss: 0.0168, norm: 1.63, time: 204.97 ms, tok/sec: 2557937.73
step: 9, loss: 0.0515, norm: 0.86, time: 205.00 ms, tok/sec: 2557548.01


In [7]:
Xb.shape

torch.Size([4, 256])

In [17]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')
cont = torch.tensor([tokenizer.encode("Why, masters,")], dtype=torch.long, device=device)
output = model.generate(cont, temp=1, max_tokens=256)
print(tokenizer.decode(output.tolist()[0]))

Why, masters, shall will! will, for will, will, will, will,
an Consety me,
Which,
AA though,
an beseech you will speaks
Unknown to the lay lay will,
an report:
Weinius,
To Cai-an and, then will!
an and return,
We have,
Say,
We will,
To the and,
They haved,
To let city city,
To by by then will,
We'll,
We are to lay the and sounds,
That living all where,
.
To our good good then,
We have have have
To make honour work?
That'sth,
On but:
CORI have
To follow
We have
To hang,
To hear,
We have,
SICINI say,
To when lay them the andinius,
,
To take
What,
To one sweet:
That's will,
rayrayray the and will will!
Your,
To hum 'ray,
To my people, let,
That they-d, and, and transmitter,---- some no better------ him?----


In [9]:
# pt clean module and _orig_mod from torch.compile() and DDP
import torch
from collections import OrderedDict

checkpoint_path = "ddp_10000.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)


state_dict = checkpoint["model"]
is_nested = True


clean_sd = OrderedDict()
for k, v in state_dict.items():
    new_key = k

    if new_key.startswith("_orig_mod."):
        new_key = new_key.replace("_orig_mod.", "", 1)

    if new_key.startswith("module."):
        new_key = new_key.replace("module.", "", 1)
    
    clean_sd[new_key] = v


print("--- Quick Key Mapping Sanity Check ---")
old_first_key = list(state_dict.keys())[0]
new_first_key = list(clean_sd.keys())[0]
print(f"Old key: {old_first_key}")
print(f"New key: {new_first_key}  <-- (Should start exactly with 'transformer...')")
print("--------------------------------------")

if is_nested:
    checkpoint["model"] = clean_sd
    final_to_save = checkpoint
else:
    final_to_save = clean_sd

new_checkpoint_path = "cleaned10k.pt"
torch.save(final_to_save, new_checkpoint_path)
print(f"Saved! Try loading '{new_checkpoint_path}' into your model now.")

--- Quick Key Mapping Sanity Check ---
Old key: _orig_mod.transformer.wte.weight
New key: transformer.wte.weight  <-- (Should start exactly with 'transformer...')
--------------------------------------
Saved! Try loading 'cleaned10k.pt' into your model now.
